# 4. Persistent cache and periodic boundary conditions

Two features concern the *lifetime* of a plan rather than a single
evaluation. The **persistent cache** stores the expensive geometry-independent
operator bank and the geometry-dependent plan on disk, so a later process
reconstructs the same plan in a fraction of the time. **Periodic boundary
conditions** evaluate the field of an infinite cubic lattice of copies of the
cell with the zero-$k$ convention.

| State | Depends on | Cached | Changes between evaluations |
|---|---|---|---|
| universal operator bank (M2M, M2L, L2L templates) | basis, order, precision | yes | never |
| periodic root operator | basis, order, precision, cell tolerance | yes | never |
| geometry plan (tree, P2M, L2P, exact near field) | normalised geometry, records, models, depth | yes | never |
| derived execution packing, device uploads | backend, packing options | no (rebuilt per process) | never |
| moments, expansion coefficients, results | the evaluate call | no | every call |

In [ ]:
import os
import tempfile
import time

import numpy as np

# The cache directory is read from the environment at every plan
# construction. This tutorial redirects it to a fresh temporary directory so
# that the cold and warm runs below are reproducible; leave it unset to use
# the build's default location.
cache_directory = tempfile.mkdtemp(prefix="cdfmm-tutorial-cache-")
os.environ["CDFMM_CACHE_DIR"] = cache_directory

import cdfmm
from tutorial_utils import (
    field_error_summary, lattice_positions, print_table, quiet_construction,
)

print("cache directory:", cache_directory)

## Part A: the persistent cache

Every problem is normalised to the root cube $[-\tfrac12, \tfrac12]^3$ before
its operators are built, so the universal bank does not depend on the physical
scale or placement and a geometry plan is shared by translated or uniformly
scaled copies of the same geometry. Files live under `<cache dir>/v1/` with
keys naming the basis, order, precision and depth, plus a SHA-256 of the
normalised geometry for plans. The environment variables are:

| Variable | Effect |
|---|---|
| `CDFMM_CACHE_DIR` | base directory (default: the build's compiled `caches/` path) |
| `CDFMM_DISABLE_CACHE=1` | force analytical reconstruction; equivalent to `options.enable_cache = False` |

A missing, truncated or incompatible file is a rebuildable miss; a cache
failure can never change a result.

In [ ]:
rng = np.random.default_rng(7)
N = 4000
positions = rng.uniform(-0.5, 0.5, size=(N, 3))
moments = rng.normal(size=(N, 3))
identities = np.arange(N, dtype=np.int32)


def make_options(cache_enabled=True):
    options = cdfmm.UniformFmmOptions()
    options.expansion_order = 6
    options.tree.max_level = 3
    options.precision = cdfmm.StaticPrecision.FLOAT32
    options.fixed_target_source_indices = identities.tolist()
    options.enable_cache = cache_enabled
    # The cache lookup/load/build seconds below are construction subphases,
    # collected only at TimingLevel.DETAILED (the default OFF reads no clock).
    options.timing_level = cdfmm.TimingLevel.DETAILED
    return options


def construct(positions, options):
    start = time.perf_counter()
    with quiet_construction():
        plan = cdfmm.UniformFmm(positions, positions, options)
    return plan, time.perf_counter() - start


cold, cold_seconds = construct(positions, make_options())
warm, warm_seconds = construct(positions, make_options())

print("universal key:", cold.universal_cache_key)
print("geometry key: ", cold.geometry_cache_key)
rows = []
for label, plan, seconds in (("cold", cold, cold_seconds), ("warm", warm, warm_seconds)):
    statistics = plan.static_plan_statistics
    rows.append({
        "run": label,
        "wall s": seconds,
        "universal hit": statistics["universal_cache_hit"],
        "geometry hit": statistics["geometry_cache_hit"],
        "bank build s": statistics["universal_operator_build_seconds"],
        "bank load s": statistics["universal_cache_load_seconds"],
        "geometry load s": statistics["geometry_cache_load_seconds"],
        "MiB read": statistics["cache_bytes_read"] / 2**20,
        "MiB written": statistics["cache_bytes_written"] / 2**20,
    })
print_table(rows, formats={"wall s": ".3f", "bank build s": ".3f", "bank load s": ".3f",
                           "geometry load s": ".3f", "MiB read": ".1f", "MiB written": ".1f"})

H_cold = cold.evaluate(moments, target_source_indices=identities)["H"]
H_warm = warm.evaluate(moments, target_source_indices=identities)["H"]
print("cold and warm fields identical:", np.array_equal(H_cold, H_warm))

The warm construction still pays for the derived execution packing and, for
FP32 plans, the precision conversion; those are not persisted because they
depend on the backend chosen in this process.

### Which geometries share a plan

The geometry key covers the normalised positions, the body records, the model
selectors, self identities, depth, periodicity, basis, order and precision.
Translating or uniformly scaling the geometry keeps the key; changing the order
or precision changes the universal key as well.

In [ ]:
translated_scaled = 2.5e-9 * positions + np.array([1.0e-9, -2.0e-9, 0.5e-9])
moved, moved_seconds = construct(translated_scaled, make_options())
print("translated+scaled geometry key equal:", moved.geometry_cache_key == cold.geometry_cache_key,
      f"(warm construction {moved_seconds:.3f} s, geometry hit "
      f"{moved.static_plan_statistics['geometry_cache_hit']})")

fp64 = make_options()
fp64.precision = cdfmm.StaticPrecision.FLOAT64
other, _ = construct(positions, fp64)
print("FP64 universal key:", other.universal_cache_key)
print("FP64 geometry key: ", other.geometry_cache_key)

uncached, uncached_seconds = construct(positions, make_options(cache_enabled=False))
print(f"\nenable_cache=False rebuilds everything: {uncached_seconds:.3f} s, "
      f"bytes read {uncached.static_plan_statistics['cache_bytes_read']}")

The installed `cdfmm-precompute` tool fills the universal files for a set of
orders and precisions without a physical problem, for example
`cdfmm-precompute --basis spherical --orders 4,6,8 --precision f32`.

## Part B: fully periodic cubic cells

Periodicity is a property of the plan. The cell must be **explicit and
cubic**; it becomes the root box, so every position must lie inside it. All
three axes are periodic. The `ZeroK0` convention omits the reciprocal $k = 0$
term and adds no macroscopic surface or demagnetising term, so an exactly
uniform magnetisation produces $H = 0$.

| Option | Meaning |
|---|---|
| `periodic.enabled` | switch the plan to the periodic model |
| `periodic.centre`, `periodic.lengths` | the cubic cell (all three lengths equal) |
| `periodic.convention` | `ZeroK0`, the only implemented convention |
| `periodic.setup_tolerance` | Ewald cut-off tolerance of the root periodiser (default $10^{-12}$) |

Inside the cell the list-1 and list-2 interaction lists wrap across the
boundaries, the 26 nearest images are traversed explicitly, and the remaining
infinite lattice is one dense root-multipole-to-root-local operator built from
an Ewald sum. Its cost is paid once and cached like the universal bank.

### What the convention means: a uniformly magnetised lattice

Take a cubic lattice of $8^3$ sites with spacing $a = L/8$ and give every site
the same moment $m = M a^3$, i.e. a uniform magnetisation $M$. Two exact
results follow from the zero-$k$ convention:

- **point dipoles at the sites** see the Lorentz local field of a cubic
  lattice, $H = +M/3$, because the cell-average field is zero and the cubic
  lattice sum adds the Lorentz term;
- **cubes of side $a$ filling the cell** see $H = 0$: each cube's own
  demagnetising field, $-M/3$, cancels the lattice term exactly.

The second statement is the "uniform magnetisation has no demagnetising
field" property of the convention; the first shows why point and finite
sources must not be mixed up when a continuum is discretised.

In [ ]:
cell_side = 1.0
sites_per_axis = 8
spacing = cell_side / sites_per_axis
lattice = lattice_positions(sites_per_axis, spacing)     # 512 cell-centred sites
lattice_identities = np.arange(len(lattice), dtype=np.int32)
cube = cdfmm.RectangularPrism(spacing, spacing, spacing)


def periodic_options(periodic, prisms=False, order=6, depth=2):
    options = cdfmm.UniformFmmOptions()
    options.precision = cdfmm.StaticPrecision.FLOAT64
    options.expansion_order = order
    options.tree.max_level = depth
    if prisms:
        options.source_geometry = cdfmm.SourceGeometry.RECTANGULAR_PRISM
        options.source_sizes = [cube]
        options.target_geometry = cdfmm.TargetGeometry.RECTANGULAR_PRISM
        options.target_sizes = [cube]
    else:
        options.fixed_target_source_indices = lattice_identities.tolist()
    if periodic:
        options.periodic.enabled = True
        options.periodic.axes = [True, True, True]
        options.periodic.centre = cdfmm.Vec3(0.0, 0.0, 0.0)
        options.periodic.lengths = cdfmm.Vec3(cell_side, cell_side, cell_side)
        options.periodic.convention = cdfmm.PeriodicConvention.ZeroK0
        options.periodic.setup_tolerance = 1.0e-12
    else:
        options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
        options.tree.root_half_width = 0.5 * cell_side
    return options


M = np.array([0.0, 0.0, 1.0])                              # uniform magnetisation
uniform_moments = np.tile(M * spacing**3, (len(lattice), 1))

periodic_plan, periodic_seconds = construct(lattice, periodic_options(True))
print(f"periodic point plan built in {periodic_seconds:.2f} s "
      f"(periodic root key {periodic_plan.periodic_cache_key})")
H_points = periodic_plan.evaluate(uniform_moments, target_source_indices=lattice_identities)["H"]
print(f"point dipoles:   mean H = {H_points.mean(axis=0)}  (M/3 = {M / 3})")

periodic_cubes, _ = construct(lattice, periodic_options(True, prisms=True))
H_cubes = periodic_cubes.evaluate(uniform_moments)["H"]
print(f"cubes filling the cell: mean H = {H_cubes.mean(axis=0)}")
print(f"site-to-site scatter (FMM truncation): {np.abs(H_points - H_points.mean(axis=0)).max():.1e}")

### Checking the periodic field against an image sum

For a non-trivial moment pattern the natural reference is the exact direct sum
over the cell and its nearest $3^3 - 1 = 26$ images, built here from
`DenseDirectPlan` copies shifted by the cell vectors. Zero total moment keeps
the omitted tail of that finite image sum small. The free-space FMM is
compared with the plain direct sum, the periodic FMM with the 27-cell sum; the
two mismatched comparisons show how much the boundary condition changes the
field.

In [ ]:
free_plan, _ = construct(lattice, periodic_options(False))
half = rng.normal(size=(len(lattice) // 2, 3))
half /= np.linalg.norm(half, axis=1)[:, None]
lattice_moments = np.concatenate((half, -half))       # zero total moment


def image_sum(radius):
    field = np.zeros((len(lattice), 3))
    for shift in np.ndindex(2 * radius + 1, 2 * radius + 1, 2 * radius + 1):
        offset = (np.asarray(shift) - radius) * cell_side
        central = not np.any(offset)
        plan = cdfmm.DenseDirectPlan(
            lattice + offset, lattice,
            target_source_indices=lattice_identities.tolist() if central else [],
            static_precision="float64",
        )
        field += plan.evaluate(lattice_moments)
    return field


H_direct_free = image_sum(0)
H_direct_periodic = image_sum(1)
H_fmm_free = free_plan.evaluate(lattice_moments, target_source_indices=lattice_identities)["H"]
H_fmm_periodic = periodic_plan.evaluate(lattice_moments, target_source_indices=lattice_identities)["H"]

rows = [
    {"comparison": "free-space FMM vs direct (1 cell)", "physics": "matching",
     "relative L2": field_error_summary(H_fmm_free, H_direct_free)["relative_l2"]},
    {"comparison": "periodic FMM vs direct (27 cells)", "physics": "matching",
     "relative L2": field_error_summary(H_fmm_periodic, H_direct_periodic)["relative_l2"]},
    {"comparison": "free-space FMM vs direct (27 cells)", "physics": "different",
     "relative L2": field_error_summary(H_fmm_free, H_direct_periodic)["relative_l2"]},
    {"comparison": "direct (27 cells) vs direct (1 cell)", "physics": "boundary effect",
     "relative L2": field_error_summary(H_direct_periodic, H_direct_free)["relative_l2"]},
]
print_table(rows, formats={"relative L2": ".3e"})

The periodic comparison carries the finite-versus-infinite image difference of
its reference as well as the FMM truncation error, so it is expected to sit a
little above the free-space one. Finite bodies work the same way: prism and
tetrahedron sources keep their exact near-field tensors for every image, and
only the singular zero-shift self pair of a point dipole is removed.

`CUDA_FULL` evaluates periodic plans field-only; choose the CPU or the hybrid
backend when the scalar potential is needed.

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)
del os.environ["CDFMM_CACHE_DIR"]
print("temporary cache removed")